# Mapa Regional — Valores Absolutos Acumulados

Dois mapas coropléticos de Minas Gerais por **Regional de Saúde** (não
município), cor de fundo contínua = quantidade acumulada de uma
variável no período inteiro disponível, com bolhas proporcionais
discretas sobrepostas indicando a quantidade acumulada de uma segunda
variável.

- **Mapa A** — cor: Casos (quantidade acumulada) · bolhas: Hospitais
  Habilitados por TIPO (ONCOLOGIA, RECONSTRUÇÃO MAMÁRIA, TRATAMENTO
  SINCRÔNICO -- retrato de 2023; TIPO="TODOS" não é usado, cada tipo
  tem sua própria bolha/cor, levemente deslocadas para não se
  esconderem).
- **Mapa B** — cor: Mortalidade (quantidade acumulada) · bolha:
  Mamografias (quantidade acumulada).

São **valores absolutos acumulados no período inteiro** — soma da
coluna `<ano>_TOTAL` de cada variável ao longo de todos os
anos disponíveis, sem nenhum ajuste por taxa/população e sem quebra
por categoria. `agregar_periodo_por_regional` (já existente em
`utilitarios_graficos_populacionais.py`) faz essa soma -- só
QUANTIDADE, sem calcular taxa (não usada por este notebook).

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))

from caminho_utilitarios import preparar_sys_path
preparar_sys_path()


PosixPath('/home/anna/tcc/Seleção e Transformação/População')

In [2]:
import json
import warnings

from IPython.display import Markdown, display

import geobr
import pandas as pd

from utilitarios_comuns import (
    COLUNA_CODIGO_IBGE,
    COLUNA_REGIONAL,
    PASTA_POPULACAO,
    PASTA_SELECAO_TRANSFORMACAO,
    ESPECIFICACOES_VARIAVEIS,
    registrar_log,
    normalizar_codigo_ibge,
    selecionar_colunas_de_valor,
    verificar_anos_unicos,
)
from utilitarios_graficos_populacionais import (
    carregar_taxa_por_regional,
    agregar_periodo_por_regional,
    classificar_em_faixas_cor,
    montar_base_mapa_coropletico_discreto,
    adicionar_camada_de_bolhas,
    adicionar_rotulos_regionais,
    adicionar_rotulos_regionais_laterais,
    montar_mapa_referencia_regionais,
    nomes_regional_originais,
    salvar_figura,
)

warnings.filterwarnings("ignore")  # avisos de CRS geográfico do geopandas, sem impacto no resultado

### Configuração

Todo parâmetro visual e de seleção de dado deste notebook está centralizado nesta célula -- nada fixado dentro das funções abaixo.

In [3]:
PASTA_TAXAS = PASTA_POPULACAO / "Taxa"
CAMINHO_MUNICIPIOS_REGIONAIS = PASTA_SELECAO_TRANSFORMACAO / "Municípios" / "tabela_municipios_regionais.csv"
PASTA_SAIDA_GRAFICOS = Path.cwd() / "Gráficos" / "Mapa Regional"

SALVAR_FIGURAS = False

ANO_MALHA_MUNICIPIOS_MG = 2024
TOLERANCIA_SIMPLIFICACAO_GEOMETRIA_METROS = 500

TOLERANCIA_SOLDA_TOPOLOGIA_METROS = 100

N_FAIXAS_COR = 5

# Bolha proporcional discreta
N_DEGRAUS_BOLHA = 5
TAMANHO_BOLHA_MINIMO = 10  # px
TAMANHO_BOLHA_MAXIMO = 34  # px
MOSTRAR_VALORES_BOLHA = False
SIMBOLO_BOLHA = "circle"  # ícone da bolha no mapa e na legenda

# Largura calculada pra caber a proporção real de MG (bounding box ~1,29:1) dentro da área
# de desenho (altura - margens t/b), sem sobrar faixa branca lateral -- antes a figura
# era mais larga que o próprio formato do estado, e o fitbounds deixava espaço vazio dos lados.
LARGURA_FIGURA = 1000
ALTURA_FIGURA = 1050

CORES_TIPO_HOSPITAL = {
    "ONCOLOGIA": "#0b1772",
    "RECONSTRUÇÃO MAMÁRIA": "#609ac3",
    "TRATAMENTO SINCRÔNICO": "#B517BB",
}

# Direção do deslocamento da bolha de cada tipo de hospital
DESLOCAMENTOS_TIPO_HOSPITAL = {
    "ONCOLOGIA": (-1.0, -0.6),
    "RECONSTRUÇÃO MAMÁRIA": (1.0, -0.6),
    "TRATAMENTO SINCRÔNICO": (0.0, 1.2),
}
DESLOCAMENTO_BOLHA_HOSPITAIS_GRAUS = 0.15  # graus de longitude/latitude

# Nome das Regionais escrito sobre os mapas A e B (o mapa de referência da última célula já traz os nomes)
MOSTRAR_NOMES_REGIONAIS = False
POSICAO_NOMES_REGIONAIS = "lado"  # "lado": colunas fora do mapa; "sobre": em cima do mapa
COR_LINHA_NOME_REGIONAL = "#4a4a4a"
DISTANCIA_NOMES_LATERAIS_GRAUS = 0.8
TAMANHO_FONTE_NOME_REGIONAL = 12
COR_NOME_REGIONAL = "#1a1a1a"
COR_CONTORNO_NOME_REGIONAL = "white"  # None = sem contorno
ESPESSURA_CONTORNO_NOME_REGIONAL = 2  # px
NOME_REGIONAL_NEGRITO = True
DESLOCAMENTO_NOME_REGIONAL_GRAUS = -0.45  # latitude; negativo = abaixo das bolhas

ESPESSURA_LINHA_CONTORNO = 1.6
COR_LINHA_CONTORNO = "#7c7777"

# Tamanhos de fonte da legenda e do título
TAMANHO_FONTE_LEGENDA_MAPA = 24
TAMANHO_FONTE_TITULO_MAPA = 24

# As legendas de cor e de bolha ficam empilhadas abaixo do mapa -- y mais espaçado pra
# caber o texto maior sem as duas linhas se encostarem
POSICAO_LEGENDA_COR = dict(orientation="h", x=0.5, xanchor="center", y=-0.08, yanchor="top")
POSICAO_LEGENDA_BOLHA = dict(orientation="h", x=0.5, xanchor="center", y=-0.20, yanchor="top")
# Sem título na figura (só legenda) -- margem superior não precisa de espaço reservado;
# margem inferior cresceu um pouco pra caber a legenda maior
MARGEM_MAPA = dict(l=40, r=40, t=30, b=300)

# Escala de cor do fundo, do mais claro ao mais escuro
ESCALA_COR = [
    "#F6E7AC",
    "#F2E33C",
    "#E8AD3F",
    "#E66C08",
    "#F30C0C",
    "#D24545",
    "#850505",
]

CONFIGURACAO_MAPA_A = {
    "VARIAVEL_COR": "Casos",
    "ESCALA_COR": ESCALA_COR,
    "TITULO": "Casos × Hospitais",
    "TITULO_LEGENDA_COR": "Número de Casos",
    "TITULO_LEGENDA_BOLHA": "Número de Hospitais",
}

CONFIGURACAO_MAPA_B = {
    "VARIAVEL_COR": "Mortalidade",
    "VARIAVEL_BOLHA": "Mamografia",
    "ESCALA_COR": ESCALA_COR,
    "COR_BOLHA": "#0b1772",
    "TITULO": "Mortalidade × Mamografias",
    "TITULO_LEGENDA_COR": "Número de Mortes",
    "TITULO_LEGENDA_BOLHA": "Número de Mamografias",
}

### Malha Geográfica — Regionais de Saúde

Municípios de MG via `geobr.read_municipality`, dissolvidos por
Regional de Saúde usando `tabela_municipios_regionais.csv`
(`REGIONAL NORMALIZADA`, já no mesmo formato de `COLUNA_REGIONAL` dos
CSVs de taxa). Municípios sem Regional mapeada recebem, apenas para
fins de geometria (sem afetar nenhum total numérico), a Regional do
vizinho mais próximo.

In [4]:
mg_municipios = geobr.read_municipality(code_muni="MG", year=ANO_MALHA_MUNICIPIOS_MG).to_crs(4326)
mg_municipios[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(mg_municipios["code_muni"])

tabela_municipios_regionais = pd.read_csv(CAMINHO_MUNICIPIOS_REGIONAIS)
tabela_municipios_regionais[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(
    tabela_municipios_regionais[COLUNA_CODIGO_IBGE]
)
tabela_municipios_regionais = tabela_municipios_regionais.rename(
    columns={"REGIONAL NORMALIZADA": COLUNA_REGIONAL}
)

mg_com_regional = mg_municipios.merge(
    tabela_municipios_regionais[[COLUNA_CODIGO_IBGE, COLUNA_REGIONAL]],
    on=COLUNA_CODIGO_IBGE, how="left",
)

municipios_sem_regional = mg_com_regional[COLUNA_REGIONAL].isna()
mg_projetado = mg_com_regional.to_crs(epsg=31983)
if municipios_sem_regional.any():
    registrar_log(
        f"{int(municipios_sem_regional.sum())} município(s) sem Regional mapeada -- "
        f"preenchendo pela Regional do vizinho mais próximo (só para fins de geometria)."
    )
    com_regional = mg_projetado[~municipios_sem_regional]
    for indice in mg_projetado[municipios_sem_regional].index:
        geometria = mg_projetado.loc[indice, "geometry"]
        distancias = com_regional.geometry.distance(geometria)
        mg_projetado.loc[indice, COLUNA_REGIONAL] = com_regional.loc[distancias.idxmin(), COLUNA_REGIONAL]

# Buffer antes do dissolve para fechar falhas de topologia entre municípios; é desfeito depois
mg_projetado["geometry"] = mg_projetado.buffer(TOLERANCIA_SOLDA_TOPOLOGIA_METROS)

geometria_regionais_projetada = mg_projetado.dissolve(by=COLUNA_REGIONAL, as_index=False)[[COLUNA_REGIONAL, "geometry"]]
geometria_regionais_projetada["geometry"] = geometria_regionais_projetada.buffer(-TOLERANCIA_SOLDA_TOPOLOGIA_METROS)

geometria_regionais = geometria_regionais_projetada.to_crs(4326)

# Nome original da Regional, só para exibição (COLUNA_REGIONAL continua normalizada, como chave de junção)
geometria_regionais["REGIONAL_EXIBICAO"] = nomes_regional_originais(geometria_regionais[COLUNA_REGIONAL])

# Centroide calculado em CRS projetado e convertido para lon/lat (WGS84), exigido pelo Scattergeo
centroides = geometria_regionais_projetada.geometry.centroid.to_crs(4326)
geometria_regionais["_LON"] = centroides.x.values
geometria_regionais["_LAT"] = centroides.y.values

# Simplifica o contorno (CRS projetado) para reduzir o GeoJSON embutido na figura
if TOLERANCIA_SIMPLIFICACAO_GEOMETRIA_METROS > 0:
    geometria_simplificada = geometria_regionais_projetada.copy()
    geometria_simplificada["geometry"] = geometria_simplificada.simplify(
        TOLERANCIA_SIMPLIFICACAO_GEOMETRIA_METROS, preserve_topology=True
    )
    geometria_simplificada = geometria_simplificada.to_crs(4326)
else:
    geometria_simplificada = geometria_regionais_projetada.to_crs(4326)

geojson_regionais = json.loads(geometria_simplificada.to_json())

registrar_log(f"Geometria dissolvida: {len(geometria_regionais)} regionais de saúde.")

[LOG] Geometria dissolvida: 28 regionais de saúde.


### Carregamento — Quantidade Acumulada por Regional

`agregar_periodo_por_regional` (já existente) soma QUANTIDADE de todas as colunas do período e devolve a QUANTIDADE acumulada por Regional -- reaproveitada aqui sem recalcular nada. Hospitais foge do padrão (retrato de 2023, TIPO="TODOS"), por isso tem um carregamento próprio, mais simples.

In [5]:
def carregar_quantidade_acumulada_periodo(nome_da_variavel):
    """Devolve a quantidade acumulada por Regional no período da variável, com o ano inicial e o final."""
    especificacao = ESPECIFICACOES_VARIAVEIS[nome_da_variavel]
    tabela_taxa_regional = carregar_taxa_por_regional(PASTA_TAXAS / especificacao["NOME_ARQUIVO_SAIDA_TAXA"])

    colunas_de_valor = selecionar_colunas_de_valor(
        tabela_taxa_regional, especificacao["ANO_INICIAL"], especificacao["ANO_FINAL"],
        colunas_a_ignorar=(COLUNA_REGIONAL,), somente_total=True,
    )
    verificar_anos_unicos(colunas_de_valor, nome_da_variavel=nome_da_variavel)

    resultado = agregar_periodo_por_regional(tabela_taxa_regional, colunas_de_valor)
    return resultado[[COLUNA_REGIONAL, "QUANTIDADE"]], especificacao["ANO_INICIAL"], especificacao["ANO_FINAL"]


def carregar_hospitais_por_tipo():
    """Devolve os hospitais em formato largo (uma coluna por TIPO, sem 'TODOS') e o ano de referência."""
    especificacao = ESPECIFICACOES_VARIAVEIS["Hospitais"]
    tabela_hospitais = carregar_taxa_por_regional(PASTA_TAXAS / especificacao["NOME_ARQUIVO_SAIDA_TAXA"])
    coluna_quantidade = f"{especificacao['ANO_INICIAL']}_TOTAL"

    tabela_tipos = tabela_hospitais.loc[
        tabela_hospitais["TIPO"] != "TODOS", [COLUNA_REGIONAL, "TIPO", coluna_quantidade]
    ]
    tabela_pivotada = tabela_tipos.pivot(index=COLUNA_REGIONAL, columns="TIPO", values=coluna_quantidade)
    return tabela_pivotada.reset_index(), especificacao["ANO_INICIAL"]


def carregar_quantidade_acumulada(nome_da_variavel):
    """Atalho para carregar_quantidade_acumulada_periodo."""
    return carregar_quantidade_acumulada_periodo(nome_da_variavel)


def formatar_periodo(ano_inicial, ano_final):
    """Texto do período: o ano, se inicial e final coincidem, ou 'inicial–final'."""
    if ano_inicial == ano_final:
        return f"{ano_inicial}"
    return f"{ano_inicial}–{ano_final}"


### Montagem do Mapa

Duas funções **novas** deste notebook: `montar_mapa_variavel_unica` (Mapa B -- cor de fundo discreta + 1 camada de bolha) e `montar_mapa_hospitais_por_tipo` (Mapa A -- cor de fundo discreta + 3 camadas de bolha, uma por TIPO de hospital, levemente deslocadas via `adicionar_camada_de_bolhas`). As duas reaproveitam `classificar_em_faixas_cor`, `montar_base_mapa_coropletico_discreto` e `adicionar_camada_de_bolhas` (novas, em `utilitarios_graficos_populacionais.py`) -- a cor de fundo vira faixas categóricas com legenda de quadradinhos (nativa de `go.Choropleth`), em vez de uma colorbar em gradiente contínuo.

In [6]:
def montar_mapa_variavel_unica(configuracao_mapa):
    """Mapa com fundo coroplético de uma variável e bolhas de outra; devolve a figura e os períodos usados."""
    nome_cor = configuracao_mapa["VARIAVEL_COR"]
    nome_bolha = configuracao_mapa["VARIAVEL_BOLHA"]

    tabela_cor, ano_ini_cor, ano_fim_cor = carregar_quantidade_acumulada(nome_cor)
    tabela_cor = tabela_cor.rename(columns={"QUANTIDADE": "QUANTIDADE_COR"})

    tabela_bolha, ano_ini_bolha, ano_fim_bolha = carregar_quantidade_acumulada(nome_bolha)
    tabela_bolha = tabela_bolha.rename(columns={"QUANTIDADE": "QUANTIDADE_BOLHA"})

    tabela_regional = (
        geometria_regionais
        .merge(tabela_cor, on=COLUNA_REGIONAL, how="left")
        .merge(tabela_bolha, on=COLUNA_REGIONAL, how="left")
    )

    rotulo_faixa, ordem_faixas, cor_por_faixa = classificar_em_faixas_cor(
        tabela_regional["QUANTIDADE_COR"], N_FAIXAS_COR, configuracao_mapa["ESCALA_COR"],
    )
    tabela_regional = tabela_regional.assign(FAIXA_COR=rotulo_faixa.values)

    periodo_cor = formatar_periodo(ano_ini_cor, ano_fim_cor)
    periodo_bolha = formatar_periodo(ano_ini_bolha, ano_fim_bolha)
    titulo_completo = configuracao_mapa["TITULO"]

    titulo_legenda_cor = configuracao_mapa.get("TITULO_LEGENDA_COR", f"Número de {nome_cor}")
    titulo_legenda_bolha = configuracao_mapa.get("TITULO_LEGENDA_BOLHA", f"Número de {nome_bolha}")

    figura = montar_base_mapa_coropletico_discreto(
        geojson=geojson_regionais, tabela_regional=tabela_regional,
        coluna_id_geojson=COLUNA_REGIONAL, coluna_regional=COLUNA_REGIONAL,
        coluna_regional_exibicao="REGIONAL_EXIBICAO",
        coluna_rotulo_faixa="FAIXA_COR", ordem_rotulos_faixa=ordem_faixas, cor_por_rotulo=cor_por_faixa,
        titulo_legenda_cor=titulo_legenda_cor,
        titulo=titulo_completo, largura=LARGURA_FIGURA, altura=ALTURA_FIGURA,
        espessura_linha_contorno=ESPESSURA_LINHA_CONTORNO, cor_linha_contorno=COR_LINHA_CONTORNO,
        posicao_legenda_cor={**POSICAO_LEGENDA_COR, "title": dict(text=titulo_legenda_cor)},
        posicao_legenda_bolha={**POSICAO_LEGENDA_BOLHA, "title": dict(text=titulo_legenda_bolha)},
        margem=MARGEM_MAPA,
        tamanho_fonte_legenda=TAMANHO_FONTE_LEGENDA_MAPA,
        tamanho_fonte_titulo=TAMANHO_FONTE_TITULO_MAPA,
    )
    if MOSTRAR_NOMES_REGIONAIS and POSICAO_NOMES_REGIONAIS == "lado":
        # Antes das bolhas, para as linhas ficarem por baixo delas
        adicionar_rotulos_regionais_laterais(
            figura, centroides_lon=tabela_regional["_LON"], centroides_lat=tabela_regional["_LAT"],
            nomes=tabela_regional["REGIONAL_EXIBICAO"], limites=geometria_regionais.total_bounds,
            tamanho_fonte_rotulo=TAMANHO_FONTE_NOME_REGIONAL, cor_texto=COR_NOME_REGIONAL,
            cor_linha=COR_LINHA_NOME_REGIONAL, distancia_borda_graus=DISTANCIA_NOMES_LATERAIS_GRAUS,
            negrito=NOME_REGIONAL_NEGRITO,
        )
    adicionar_camada_de_bolhas(
        figura, tabela_regional=tabela_regional, coluna_regional=COLUNA_REGIONAL,
        coluna_regional_exibicao="REGIONAL_EXIBICAO",
        coluna_valor="QUANTIDADE_BOLHA", cor_bolha=configuracao_mapa["COR_BOLHA"],
        nome_serie_bolha=titulo_legenda_bolha, mostrar_titulo_grupo=False,
        centroides_lon=tabela_regional["_LON"], centroides_lat=tabela_regional["_LAT"],
        n_degraus=N_DEGRAUS_BOLHA, tamanho_minimo=TAMANHO_BOLHA_MINIMO, tamanho_maximo=TAMANHO_BOLHA_MAXIMO,
        mostrar_valores=MOSTRAR_VALORES_BOLHA, simbolo_bolha=SIMBOLO_BOLHA, nome_legenda="legend2",
    )
    if MOSTRAR_NOMES_REGIONAIS and POSICAO_NOMES_REGIONAIS == "sobre":
        adicionar_rotulos_regionais(
            figura, centroides_lon=tabela_regional["_LON"], centroides_lat=tabela_regional["_LAT"],
            nomes=tabela_regional["REGIONAL_EXIBICAO"], tamanho_fonte_rotulo=TAMANHO_FONTE_NOME_REGIONAL,
            cor_texto=COR_NOME_REGIONAL, cor_contorno=COR_CONTORNO_NOME_REGIONAL,
            espessura_contorno=ESPESSURA_CONTORNO_NOME_REGIONAL, negrito=NOME_REGIONAL_NEGRITO,
            deslocamento_lat=DESLOCAMENTO_NOME_REGIONAL_GRAUS,
        )
    return figura, periodo_cor, periodo_bolha


def montar_mapa_hospitais_por_tipo(configuracao_mapa):
    """Mapa com fundo coroplético de uma variável e uma camada de bolhas por tipo de hospital."""
    nome_cor = configuracao_mapa["VARIAVEL_COR"]

    tabela_cor, ano_ini_cor, ano_fim_cor = carregar_quantidade_acumulada(nome_cor)
    tabela_cor = tabela_cor.rename(columns={"QUANTIDADE": "QUANTIDADE_COR"})

    tabela_hospitais_tipo, ano_hospitais = carregar_hospitais_por_tipo()

    tabela_regional = (
        geometria_regionais
        .merge(tabela_cor, on=COLUNA_REGIONAL, how="left")
        .merge(tabela_hospitais_tipo, on=COLUNA_REGIONAL, how="left")
    )

    rotulo_faixa, ordem_faixas, cor_por_faixa = classificar_em_faixas_cor(
        tabela_regional["QUANTIDADE_COR"], N_FAIXAS_COR, configuracao_mapa["ESCALA_COR"],
    )
    tabela_regional = tabela_regional.assign(FAIXA_COR=rotulo_faixa.values)

    periodo_cor = formatar_periodo(ano_ini_cor, ano_fim_cor)
    titulo_completo = configuracao_mapa["TITULO"]

    titulo_legenda_cor = configuracao_mapa.get("TITULO_LEGENDA_COR", f"Número de {nome_cor}")
    titulo_legenda_bolha = configuracao_mapa.get("TITULO_LEGENDA_BOLHA", "Número de Hospitais")

    figura = montar_base_mapa_coropletico_discreto(
        geojson=geojson_regionais, tabela_regional=tabela_regional,
        coluna_id_geojson=COLUNA_REGIONAL, coluna_regional=COLUNA_REGIONAL,
        coluna_regional_exibicao="REGIONAL_EXIBICAO",
        coluna_rotulo_faixa="FAIXA_COR", ordem_rotulos_faixa=ordem_faixas, cor_por_rotulo=cor_por_faixa,
        titulo_legenda_cor=titulo_legenda_cor,
        titulo=titulo_completo, largura=LARGURA_FIGURA, altura=ALTURA_FIGURA,
        espessura_linha_contorno=ESPESSURA_LINHA_CONTORNO, cor_linha_contorno=COR_LINHA_CONTORNO,
        posicao_legenda_cor={**POSICAO_LEGENDA_COR, "title": dict(text=titulo_legenda_cor)},
        posicao_legenda_bolha={
            **POSICAO_LEGENDA_BOLHA, "title": dict(text=titulo_legenda_bolha),
        },
        margem=MARGEM_MAPA,
        tamanho_fonte_legenda=TAMANHO_FONTE_LEGENDA_MAPA,
        tamanho_fonte_titulo=TAMANHO_FONTE_TITULO_MAPA,
    )

    if MOSTRAR_NOMES_REGIONAIS and POSICAO_NOMES_REGIONAIS == "lado":
        # Antes das bolhas, para as linhas ficarem por baixo delas
        adicionar_rotulos_regionais_laterais(
            figura, centroides_lon=tabela_regional["_LON"], centroides_lat=tabela_regional["_LAT"],
            nomes=tabela_regional["REGIONAL_EXIBICAO"], limites=geometria_regionais.total_bounds,
            tamanho_fonte_rotulo=TAMANHO_FONTE_NOME_REGIONAL, cor_texto=COR_NOME_REGIONAL,
            cor_linha=COR_LINHA_NOME_REGIONAL, distancia_borda_graus=DISTANCIA_NOMES_LATERAIS_GRAUS,
            negrito=NOME_REGIONAL_NEGRITO,
        )

    for tipo, deslocamento in DESLOCAMENTOS_TIPO_HOSPITAL.items():
        if tipo not in tabela_regional.columns:
            registrar_log(f"ATENÇÃO: TIPO '{tipo}' não encontrado em hospitais_taxa.csv -- camada de bolha ignorada.")
            continue
        adicionar_camada_de_bolhas(
            figura, tabela_regional=tabela_regional, coluna_regional=COLUNA_REGIONAL,
            coluna_regional_exibicao="REGIONAL_EXIBICAO",
            coluna_valor=tipo, cor_bolha=CORES_TIPO_HOSPITAL[tipo], nome_serie_bolha=tipo.title(),
            centroides_lon=tabela_regional["_LON"], centroides_lat=tabela_regional["_LAT"],
            n_degraus=N_DEGRAUS_BOLHA, tamanho_minimo=TAMANHO_BOLHA_MINIMO, tamanho_maximo=TAMANHO_BOLHA_MAXIMO,
            deslocamento_lon=deslocamento[0] * DESLOCAMENTO_BOLHA_HOSPITAIS_GRAUS,
            deslocamento_lat=deslocamento[1] * DESLOCAMENTO_BOLHA_HOSPITAIS_GRAUS,
            mostrar_valores=MOSTRAR_VALORES_BOLHA, simbolo_bolha=SIMBOLO_BOLHA, nome_legenda="legend2",
        )

    if MOSTRAR_NOMES_REGIONAIS and POSICAO_NOMES_REGIONAIS == "sobre":
        adicionar_rotulos_regionais(
            figura, centroides_lon=tabela_regional["_LON"], centroides_lat=tabela_regional["_LAT"],
            nomes=tabela_regional["REGIONAL_EXIBICAO"], tamanho_fonte_rotulo=TAMANHO_FONTE_NOME_REGIONAL,
            cor_texto=COR_NOME_REGIONAL, cor_contorno=COR_CONTORNO_NOME_REGIONAL,
            espessura_contorno=ESPESSURA_CONTORNO_NOME_REGIONAL, negrito=NOME_REGIONAL_NEGRITO,
            deslocamento_lat=DESLOCAMENTO_NOME_REGIONAL_GRAUS,
        )

    return figura, periodo_cor, ano_hospitais

### Mapa A — Casos × Hospitais Habilitados por Tipo

In [7]:
figura_mapa_a, periodo_cor_a, ano_hospitais_a = montar_mapa_hospitais_por_tipo(CONFIGURACAO_MAPA_A)
figura_mapa_a.show()

display(Markdown(
    f"**Nota:** cor de fundo = Casos acumulados ({periodo_cor_a}); bolhas = Hospitais "
    f"Habilitados por TIPO (ONCOLOGIA, RECONSTRUÇÃO MAMÁRIA, TRATAMENTO SINCRÔNICO -- "
    f"retrato de {ano_hospitais_a}, não um acumulado de período; TIPO=\"TODOS\" não é usado)."
))

if SALVAR_FIGURAS:
    salvar_figura(figura_mapa_a, "mapa_regional_casos_hospitais", PASTA_SAIDA_GRAFICOS)

**Nota:** cor de fundo = Casos acumulados (1985–2023); bolhas = Hospitais Habilitados por TIPO (ONCOLOGIA, RECONSTRUÇÃO MAMÁRIA, TRATAMENTO SINCRÔNICO -- retrato de 2023, não um acumulado de período; TIPO="TODOS" não é usado).

### Mapa B — Mortalidade × Mamografias

In [8]:
figura_mapa_b, periodo_cor_b, periodo_bolha_b = montar_mapa_variavel_unica(CONFIGURACAO_MAPA_B)
figura_mapa_b.show()

display(Markdown(
    f"**Nota:** cor de fundo = Mortalidade acumulada ({periodo_cor_b}); bolha = Mamografias "
    f"acumuladas ({periodo_bolha_b})."
))

if SALVAR_FIGURAS:
    salvar_figura(figura_mapa_b, "mapa_regional_mortalidade_mamografia", PASTA_SAIDA_GRAFICOS)

**Nota:** cor de fundo = Mortalidade acumulada (1979–2024); bolha = Mamografias acumuladas (2013–2025).

In [9]:
print("===== Valores usados no Mapa A (Casos x Hospitais por Tipo) =====")
tabela_cor_a, ano_ini_cor_a, ano_fim_cor_a = carregar_quantidade_acumulada(CONFIGURACAO_MAPA_A["VARIAVEL_COR"])
tabela_cor_a = tabela_cor_a.rename(columns={"QUANTIDADE": "QUANTIDADE_COR_CASOS"})
tabela_hospitais_tipo, ano_hospitais = carregar_hospitais_por_tipo()
tabela_a = tabela_cor_a.merge(tabela_hospitais_tipo, on=COLUNA_REGIONAL, how="left")
print(f"Casos: {ano_ini_cor_a}-{ano_fim_cor_a} | Hospitais: retrato de {ano_hospitais}")
display(tabela_a.sort_values(COLUNA_REGIONAL).reset_index(drop=True))

print("\n===== Valores usados no Mapa B (Mortalidade x Mamografia) =====")
tabela_cor_b, ano_ini_cor_b, ano_fim_cor_b = carregar_quantidade_acumulada(CONFIGURACAO_MAPA_B["VARIAVEL_COR"])
tabela_cor_b = tabela_cor_b.rename(columns={"QUANTIDADE": "QUANTIDADE_COR_MORTALIDADE"})
tabela_bolha_b, ano_ini_bolha_b, ano_fim_bolha_b = carregar_quantidade_acumulada(CONFIGURACAO_MAPA_B["VARIAVEL_BOLHA"])
tabela_bolha_b = tabela_bolha_b.rename(columns={"QUANTIDADE": "QUANTIDADE_BOLHA_MAMOGRAFIA"})
tabela_b = tabela_cor_b.merge(tabela_bolha_b, on=COLUNA_REGIONAL, how="left")
print(f"Mortalidade: {ano_ini_cor_b}-{ano_fim_cor_b} | Mamografia: {ano_ini_bolha_b}-{ano_fim_bolha_b}")
display(tabela_b.sort_values(COLUNA_REGIONAL).reset_index(drop=True))

===== Valores usados no Mapa A (Casos x Hospitais por Tipo) =====
Casos: 1985-2023 | Hospitais: retrato de 2023


,REGIONAL DE SAÚDE,QUANTIDADE_COR_CASOS,ONCOLOGIA,RECONSTRUÇÃO MAMÁRIA,TRATAMENTO SINCRÔNICO
0,alfenas,3324.0,1.0,1.0,NaN
1,barbacena,3615.0,1.0,NaN,NaN
2,belo horizonte,35740.0,8.0,6.0,NaN
3,coronel fabriciano,4112.0,1.0,NaN,NaN
4,diamantina,1359.0,NaN,NaN,NaN
5,divinopolis,8153.0,2.0,2.0,NaN
6,governador valadares,3111.0,1.0,1.0,NaN
7,itabira,2622.0,1.0,1.0,NaN
8,ituiutaba,973.0,NaN,NaN,NaN
9,januaria,774.0,NaN,NaN,NaN



===== Valores usados no Mapa B (Mortalidade x Mamografia) =====
Mortalidade: 1979-2024 | Mamografia: 2013-2025


,REGIONAL DE SAÚDE,QUANTIDADE_COR_MORTALIDADE,QUANTIDADE_BOLHA_MAMOGRAFIA
0,alfenas,985.0,107923.0
1,barbacena,1071.0,122795.0
2,belo horizonte,14509.0,964945.0
3,coronel fabriciano,1496.0,150827.0
4,diamantina,463.0,77357.0
5,divinopolis,2684.0,205687.0
6,governador valadares,1319.0,122940.0
7,itabira,847.0,86811.0
8,ituiutaba,385.0,32422.0
9,januaria,299.0,49872.0


### Mapa de Referência — Nomes das Regionais de Saúde

Sem nenhum dado nem bolha -- só a malha das 28 Regionais numa cor
única (azul acinzentado) com o nome de cada uma escrito em azul
marinho sobre a própria área. Serve de referência visual para localizar
cada Regional citada nos Mapas A e B, que não têm mais os nomes por cima.

In [10]:
COR_PREENCHIMENTO_MAPA_NOMES = "#8FC9A4"
COR_NOME_REGIONAL_MAPA_NOMES = "#17422B"
COR_LINHA_CONTORNO_MAPA_NOMES = "white"
COR_LINHA_CONTORNO_EXTERNA_MAPA_NOMES = "#4C8562"
ESPESSURA_LINHA_CONTORNO_MAPA_NOMES = 2.2
ESPESSURA_LINHA_CONTORNO_EXTERNA_MAPA_NOMES = 3
TAMANHO_FONTE_NOME_MAPA_NOMES = 18
TAMANHO_FONTE_NOME_MINIMO_MAPA_NOMES = 11
# Largura ajustada pra bater com a proporção real de MG dentro da área de desenho (mesmo
# ajuste do mapa principal), sem sobrar faixa branca lateral.
LARGURA_MAPA_NOMES = 1800
ALTURA_MAPA_NOMES = 1400

figura_mapa_nomes = montar_mapa_referencia_regionais(
    geojson=geojson_regionais, geometria_regionais=geometria_regionais,
    coluna_id_geojson=COLUNA_REGIONAL, coluna_regional_exibicao="REGIONAL_EXIBICAO",
    titulo="",
    cor_preenchimento=COR_PREENCHIMENTO_MAPA_NOMES, cor_nome=COR_NOME_REGIONAL_MAPA_NOMES,
    cor_linha_contorno=COR_LINHA_CONTORNO_MAPA_NOMES,
    cor_linha_contorno_externa=COR_LINHA_CONTORNO_EXTERNA_MAPA_NOMES,
    espessura_linha_contorno_externa=ESPESSURA_LINHA_CONTORNO_EXTERNA_MAPA_NOMES,
    largura=LARGURA_MAPA_NOMES, altura=ALTURA_MAPA_NOMES,
    espessura_linha_contorno=ESPESSURA_LINHA_CONTORNO_MAPA_NOMES, tamanho_fonte_nome=TAMANHO_FONTE_NOME_MAPA_NOMES,
    tamanho_fonte_nome_minimo=TAMANHO_FONTE_NOME_MINIMO_MAPA_NOMES,
    tamanho_fonte_titulo=TAMANHO_FONTE_TITULO_MAPA, fundo_transparente=True,
)
figura_mapa_nomes.show()

if SALVAR_FIGURAS:
    salvar_figura(figura_mapa_nomes, "mapa_regional_nomes_referencia", PASTA_SAIDA_GRAFICOS)
